# Performance: manual collection + cleaning
Stats from WhoScored (primary) and FBref (cross-check),
domestic league 2024/25 to build `performance.csv`.

In [1]:
import os
BASE = os.path.abspath('dissertation_data')
os.makedirs(BASE, exist_ok=True)

In [2]:
# Configuration
FRAME_CSV = os.path.join(BASE, 'sample_frame.csv')
STATS_CSV = os.path.join(BASE, 'player_stats_manual.csv')
PERF_CSV  = os.path.join(BASE, 'performance.csv')
DISCREP_CSV = os.path.join(BASE, 'data_discrepancy_log.csv')
REPORT_TXT  = os.path.join(BASE, 'data_quality_report.txt')

In [3]:
# Helpers
import re, unicodedata
import pandas as pd, numpy as np
def norm_name(name):
    if not isinstance(name, str): return ''
    s = unicodedata.normalize('NFKD', name).encode('ascii','ignore').decode()
    s = re.sub(r'[^a-zA-Z ]', '', s).strip().lower()
    return re.sub(r'\s+', ' ', s)
def classify_position(raw):
    if not isinstance(raw, str) or not raw.strip(): return None
    s = re.split(r'[,/]', raw.strip().lower())[0].strip()
    if s=='gk' or 'goalkeep' in s or 'keeper' in s: return 'Goalkeeper'
    if 'midfield' in s or s in ('mf','cm','dm','am','cdm','cam'): return 'Midfielder'
    if 'back' in s or 'defen' in s or s=='df' or s in ('cb','rb','lb','rwb','lwb'): return 'Defender'
    if ('forward' in s or 'wing' in s or 'strik' in s or 'attack' in s or s=='fw'
            or s in ('cf','ss','rw','lw','st')): return 'Forward'
    return None

## Mid-season transfers
Stats are recorded at the END-of-season club (e.g. Kvaratskhelia → PSG).
`club_season_start` keeps the starting club and `mid_season_tranfers` = 1 flags the move.

In [6]:
assert os.path.exists(FRAME_CSV), 'sample_frame.csv not found'
frame = pd.read_csv(FRAME_CSV)

# Column groups — MUST match player_stats_manual.csv exactly
ID    = ['club_2024_25','league_2024_25','position','nationality','club_season_start','mid_season_tranfers']
# WhoScored line: MP/Min/Gls/Ast are season totals; the rest are WhoScored per-game averages
WS    = ['MP','Min','Gls','Ast','shots_pg','dribbles','fouled','offsides','dispossessed',
         'key_passes','passes_pg','pass_pct','crosses','long_balls',
         'tackles','interceptions','fouls','clearances','blocks']
FBREF = ['fbref_MP','fbref_Gls','fbref_Ast']                 # FBref cross-check (MP, goals, assists)
GK    = ['gk_goals_against','gk_save_pct','gk_clean_sheets']  # goalkeepers only (from FBref)
TEAM  = ['team_league_position','ucl_stage','Whoscored_Rating']
STAT_COLS = ID + WS + FBREF + GK + TEAM

if not os.path.exists(STATS_CSV):
    t = frame[['player_name']].copy()
    t['club_2024_25']       = frame.get('bdor_club','')          # Ballon d'Or clubs pre-filled as a hint
    t['league_2024_25']     = ''
    t['position']           = ''
    t['nationality']        = frame.get('bdor_nationality','')
    t['club_season_start']  = ''   # club they STARTED the season at (explanatory; e.g. Napoli)
    t['mid_season_tranfers']= ''   # 1 if the player changed club mid-2024/25 (else 0)
    for col in WS+FBREF+GK+TEAM: t[col] = ''
    t = t[['player_name'] + STAT_COLS]
    t.to_csv(STATS_CSV, index=False)
    print('Wrote template:', STATS_CSV)
    print('Fill it in (domestic league, 2024/25), then re-run.')
else:
    print('Template already exists:', STATS_CSV)

Template already exists: /content/dissertation_data/player_stats_manual.csv


## Clean & validate the filled template
Coerces text to numbers, trims names, runs range/consistency checks, cross-checks FBref vs WhoScored
fundamentals into a discrepancy log, and report

In [7]:
stats = pd.read_csv(STATS_CSV)
stats.columns = [c.strip() for c in stats.columns]
stats['name_key'] = stats['player_name'].map(norm_name)

EMPTY = ['','nan','NaN','None','-','\u2014','n/a','N/A']    # markers that mean "no data"
NUMERIC = ['MP','Min','Gls','Ast','shots_pg','dribbles','fouled','offsides','dispossessed',
           'key_passes','passes_pg','pass_pct','crosses','long_balls',
           'tackles','interceptions','fouls','clearances','blocks',
           'fbref_MP','fbref_Gls','fbref_Ast',
           'gk_goals_against','gk_save_pct','gk_clean_sheets','Whoscored_Rating']
issues = []
for col in NUMERIC:
    if col in stats.columns:
        cleaned = (stats[col].astype(str).str.replace('%','',regex=False)
                   .str.replace(',','',regex=False).str.strip()).replace(EMPTY, np.nan)
        num = pd.to_numeric(cleaned, errors='coerce')
        bad = stats['player_name'][cleaned.notna() & num.isna()].tolist()
        if bad: issues.append(f"[{col}] non-numeric: {', '.join(bad[:8])}" + (' ...' if len(bad)>8 else ''))
        stats[col] = num

# league position entered as ordinals ('1st','2nd') -> integer
if 'team_league_position' in stats.columns:
    stats['team_league_position'] = pd.to_numeric(
        stats['team_league_position'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

stats['position_group'] = stats['position'].map(classify_position)
unresolved = stats.loc[stats['position_group'].isna() & stats['position'].notna(), 'player_name'].tolist()

def flag(mask, msg):
    names = stats['player_name'][mask.fillna(False)].tolist()
    if names: issues.append(f"{msg}: {', '.join(names[:8])}" + (' ...' if len(names)>8 else ''))
flag((stats.get('Min',1)<200) & (stats.get('MP',0)>3), 'Min < 200 but MP > 3 (likely a typo)')
flag(stats['club'].astype(str).str.contains(r'League|Liga|Serie|Ligue|Bundesliga',case=False,na=False)
     if 'club' in stats else stats['club_2024_25'].astype(str).str.contains(r'League|Liga|Serie|Ligue|Bundesliga',case=False,na=False),
     'club looks like a league name (check)')
if 'pass_pct' in stats: flag((stats['pass_pct']<0)|(stats['pass_pct']>100), 'pass_pct out of 0-100')
if 'gk_save_pct' in stats: flag((stats['gk_save_pct']<0)|(stats['gk_save_pct']>100), 'gk_save_pct out of 0-100')
for col in ['Gls','Ast']:
    if col in stats: flag(stats[col].notna() & (stats[col]%1!=0), f'{col} not a whole number (check)')

gk = stats['position_group']=='Goalkeeper'
gk_missing = stats['player_name'][gk & stats[['gk_save_pct','gk_goals_against']].isna().all(axis=1)].tolist()
if gk_missing: issues.append('goalkeepers missing GK stats: '+', '.join(gk_missing[:8]))

blank_core = stats['player_name'][stats[['Min','Gls','Ast']].isna().any(axis=1) | (stats['position'].fillna('')=='')].tolist()
print('=== DATA QUALITY ==='); print('Players:', len(stats)); print('Issues found:', len(issues))
for s in issues: print('  -', s)
if unresolved: print('  - position not understood:', ', '.join(unresolved[:8]))
print('Rows missing core fields (no WhoScored coverage):', len(blank_core), '->', ', '.join(blank_core[:8]))

=== DATA QUALITY ===
Players: 146
Issues found: 0
Rows missing core fields (no WhoScored coverage): 1 -> Cristiano Ronaldo


In [9]:
# Cross-check — WhoScored is the RECORDED value, FBref fills only blanks, differences logged
stats['goals']       = stats['Gls'].combine_first(stats['fbref_Gls'])
stats['assists']     = stats['Ast'].combine_first(stats['fbref_Ast'])
stats['appearances'] = stats['MP'].combine_first(stats['fbref_MP'])
stats['minutes']     = stats['Min']
gd = (stats['fbref_Gls'] - stats['Gls']).abs()
ad = (stats['fbref_Ast'] - stats['Ast']).abs()
md = (stats['fbref_MP']  - stats['MP']).abs()
disc_mask = (pd.concat([gd,ad,md],axis=1).fillna(0) > 0).any(axis=1)
disc = stats.loc[disc_mask, ['player_name','club','Gls','fbref_Gls','Ast','fbref_Ast','MP','fbref_MP']].copy() \
       if 'club' in stats.columns else \
       stats.loc[disc_mask, ['player_name','club_2024_25','Gls','fbref_Gls','Ast','fbref_Ast','MP','fbref_MP']].copy()
disc['resolution'] = ''
disc['resolution_source'] = ''
disc.to_csv(DISCREP_CSV, index=False)
print('Discrepancy log:', len(disc), 'rows ->', DISCREP_CSV)

# derived — goals/assists are season totals -> per-90. the WhoScored detail stats are already per-game
n90 = (stats['minutes']/90).replace(0, np.nan)
stats['G_plus_A'] = stats['goals'].fillna(0) + stats['assists'].fillna(0)
for col in ['goals','assists']:
    stats[col+'_p90'] = (stats[col]/n90).round(3)

# Champions League stage -> ordinal (further = higher). League Phase kept distinct from Not Qualified.
UCL = {'not qualified':0,'league phase':1,'group':1,
       'knockout phase play-offs':2,'knockout phase play offs':2,'play-offs':2,'playoffs':2,
       'round of 16':3,'r16':3,'quarter final':4,'quarter-final':4,'qf':4,
       'semi final':5,'semi-final':5,'sf':5,'final':6,'winner':7,'winners':7}
stats['ucl_stage_ord'] = stats['ucl_stage'].astype(str).str.lower().str.strip().map(UCL)
unmapped = sorted(set(stats.loc[stats['ucl_stage_ord'].isna() & stats['ucl_stage'].notna(),'ucl_stage'].astype(str)))
if unmapped: print('  ! ucl_stage not recognised (add to UCL map):', unmapped)

with open(REPORT_TXT,'w') as f:
    f.write('DATA QUALITY REPORT\n===================\n')
    f.write(f'Players: {len(stats)}\nDiscrepancies (FBref vs WhoScored): {len(disc)}\n\n')
    for s in issues: f.write('- '+s+'\n')
print('Quality report ->', REPORT_TXT)

Discrepancy log: 4 rows -> /content/dissertation_data/data_discrepancy_log.csv
Quality report -> /content/dissertation_data/data_quality_report.txt


In [11]:
# FINAL
frame_small = frame[['name_key','player_name','in_ballondor','consensus_count']]
out = frame_small.merge(stats.drop(columns=['player_name']), on='name_key', how='left')
out = out.rename(columns={'club_2024_25':'club','league_2024_25':'league'})

FINAL = ['player_name','name_key','in_ballondor','consensus_count',
         'club','league','position','position_group','nationality','club_season_start','mid_season_tranfers',
         'appearances','minutes','goals','assists','G_plus_A','goals_p90','assists_p90',
         'shots_pg','dribbles','fouled','offsides','dispossessed','key_passes','passes_pg','pass_pct',
         'crosses','long_balls','tackles','interceptions','fouls','clearances','blocks',
         'gk_goals_against','gk_save_pct','gk_clean_sheets',
         'team_league_position','ucl_stage','ucl_stage_ord','Whoscored_Rating']
out = out[[c for c in FINAL if c in out.columns]]
out.to_csv(PERF_CSV, index=False)
print('Wrote FINAL', PERF_CSV, '(', len(out), 'players,', out.shape[1], 'columns )')
print('Dropped the raw WhoScored/FBref duplicates + qc columns (kept in the discrepancy log).')
out.head()

Wrote FINAL /content/dissertation_data/performance.csv ( 146 players, 40 columns )
Dropped the raw WhoScored/FBref duplicates + qc columns (kept in the discrepancy log).


,player_name,name_key,in_ballondor,consensus_count,club,league,position,position_group,nationality,club_season_start,...,fouls,clearances,blocks,gk_goals_against,gk_save_pct,gk_clean_sheets,team_league_position,ucl_stage,ucl_stage_ord,Whoscored_Rating
0,Ousmane Dembélé,ousmane dembele,True,3,Paris Saint-Germain,Ligue 1,Forward,Forward,France,Paris Saint-Germain,...,0.3,0.0,0.0,NaN,NaN,NaN,1,Winner,7,7.65
1,Lamine Yamal,lamine yamal,True,3,Barcelona,La Liga,Forward,Forward,Spain,Barcelona,...,0.9,0.1,0.0,NaN,NaN,NaN,1,Semi Final,5,8.01
2,Vitinha,vitinha,True,3,Paris Saint-Germain,Ligue 1,Midfielder,Midfielder,Portugal,Paris Saint-Germain,...,0.3,0.6,0.2,NaN,NaN,NaN,1,Winner,7,7.01
3,Mohamed Salah,mohamed salah,True,3,Liverpool,Premier League,Forward,Forward,Egypt,Liverpool,...,0.7,0.1,0.0,NaN,NaN,NaN,1,Round of 16,3,7.61
4,Raphinha,raphinha,True,3,Barcelona,La Liga,Forward,Forward,Brazil,Barcelona,...,0.3,0.6,0.1,NaN,NaN,NaN,1,Semi Final,5,7.62
